# Notebook 07: Repeated Comparison of 3 LoRA Architectures (Patch 8)

We repeat each experiment **10 times** (different random seeds) to obtain reliable mean ± std estimates for 4 models on the LSST dataset:

1. **FullFT p8 + Mean Pooling**: Single encoder (patch 8, fully fine-tuned). Mean pooling over patches per variable → Linear classifier. Serves as our reference baseline.

2. **FullFT p8 + MHA (4 heads)**: Single encoder (patch 8, fully fine-tuned). A learnable CLS token per variable attends over the temporal patches via Multi-Head Attention (4 heads) → Linear classifier.

3. **Dual LoRA (p64 + p8) + Cascade Cross-Attention**: Two independent LoRA encoders.
   - **Coarse encoder** (patch 64, LoRA r=8): We keep only the **first patch embedding** per variable — a compact coarse-grained summary.
   - **Fine encoder** (patch 8, LoRA r=8): All patch embeddings per variable — fine-grained temporal details.
   - **Cascade**: The coarse embedding acts as the **query** in a cross-attention layer where the fine embeddings are key/value. This enriches the coarse representation with fine-grained context before classification.

4. **LoRA p64 + FullFT p8 + Cascade**: Same cascade architecture as Model 3, but asymmetric training — coarse encoder uses LoRA r=8, fine encoder is fully fine-tuned.

In [38]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from IPython.display import display

from peft import get_peft_model, LoraConfig
from uni2ts.model.moirai import MoiraiModule

from encoder import MoiraiEncoder
from heads import MeanPoolingClassifier, SingleScaleMultiHeadClassifier
from models import FullHeadWrapper, LoraHeadWrapper
from utils import get_lsst_dataloaders, universal_grid_search, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
BATCH_SIZE = 256

LORA_R = 8
NUM_HEADS_MHA = 4

lr_grid = [5*1e-5]
wd_grid = [0.05, 0.01]

SEEDS = [42, 1337, 0, 7, 99, 123, 256, 512, 2024, 314]
N_REPEATS = len(SEEDS)

print(f"Device: {DEVICE}")
print(f"Seeds: {SEEDS}")

Device: cuda
Seeds: [42, 1337, 0, 7, 99, 123, 256, 512, 2024, 314]


In [39]:
# Load data once (train/val split is fixed via random_state=42 in utils)
set_seed(42)
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)
print(f"Number of classes: {num_classes}")

Number of classes: 14


## Model 3 Definition: Dual Encoder — FullFT p8 + LoRA p64 — Mean Pooling + Concat

Two Moirai encoders with asymmetric fine-tuning. Each encoder produces patch embeddings per variable; mean pooling is applied over patches independently for each encoder and each variable. The two mean-pooled representations are concatenated and fed into a linear classifier.

- **Fine encoder** (patch 8) → fully fine-tuned
- **Coarse encoder** (patch 64) → LoRA r=8

In [40]:
class DualHybridMeanPoolWrapper(nn.Module):
    """
    Dual encoder mean pooling: FullFT p8 + LoRA p64.

    Architecture:
      1. Fine encoder (patch 8, fully fine-tuned):
         Produces Z_fine of shape (B, num_vars * P_fine, F).
         Mean pooling over P_fine patches per variable → (B, num_vars, F).

      2. Coarse encoder (patch 64, LoRA r=lora_r):
         Produces Z_coarse of shape (B, num_vars * P_coarse, F).
         Mean pooling over P_coarse patches per variable → (B, num_vars, F).

      3. Concatenate fine and coarse mean-pooled representations per variable:
         → (B, num_vars, 2*F), flattened to (B, num_vars * 2 * F).

      4. Dropout → Linear → num_classes.
    """
    def __init__(self, num_classes, num_vars=6, size="small", lora_r=8, in_features=384):
        super().__init__()
        self.num_vars = num_vars
        self.in_features = in_features

        # --- Fine encoder: patch_size = 8, fully fine-tuned ---
        self.encoder_fine = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=8, context_length=36, patch_size=8,
            num_samples=100, target_dim=num_vars,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        # All parameters trainable by default

        # --- Coarse encoder: patch_size = 64, LoRA ---
        enc_coarse = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=64, context_length=36, patch_size=64,
            num_samples=100, target_dim=num_vars,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        self.encoder_coarse = get_peft_model(enc_coarse, LoraConfig(
            r=lora_r, lora_alpha=lora_r * 2, target_modules="all-linear",
        ))

        # --- Classifier head ---
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(num_vars * 2 * in_features, num_classes)

    def forward(self, target, obs, pad):
        B = target.shape[0]
        F = self.in_features

        # 1. Fine encoder → mean pool over patches per variable
        Z_fine = self.encoder_fine(target, obs, pad)          # (B, num_vars*P_fine, F)
        P_fine = Z_fine.shape[1] // self.num_vars
        Z_fine = Z_fine.view(B, self.num_vars, P_fine, F)
        fine_pool = Z_fine.mean(dim=2)                        # (B, num_vars, F)

        # 2. Coarse encoder → mean pool over patches per variable
        Z_coarse = self.encoder_coarse(target, obs, pad)      # (B, num_vars*P_coarse, F)
        P_coarse = Z_coarse.shape[1] // self.num_vars
        Z_coarse = Z_coarse.view(B, self.num_vars, P_coarse, F)
        coarse_pool = Z_coarse.mean(dim=2)                    # (B, num_vars, F)

        # 3. Concatenate along feature dim, flatten
        combined = torch.cat([fine_pool, coarse_pool], dim=2) # (B, num_vars, 2*F)
        out = combined.view(B, -1)                            # (B, num_vars * 2 * F)

        # 4. Classify
        return self.classifier(self.dropout(out))

## Model 4 Definition: Hybrid Dual Encoder — LoRA p64 + FullFT p8

Same cascade cross-attention architecture as Model 3, but with asymmetric training:
- **Coarse encoder** (patch 64) → LoRA r=8 (parameter-efficient)
- **Fine encoder** (patch 8) → Full fine-tuning (all parameters trainable)

In [41]:
class DualHybridCoarseToFineWrapper(nn.Module):
    """
    Hybrid dual encoder cascade: LoRA p64 (coarse) + FullFT p8 (fine).

    Architecture identical to DualLoraCoarseToFineWrapper, but:
      - Coarse encoder (patch 64) uses LoRA r=lora_r (frozen base + adapters)
      - Fine encoder  (patch 8)  is fully fine-tuned (all weights trainable)
    """
    def __init__(self, num_classes, num_vars=6, size="small", lora_r=8, num_heads=4, in_features=384):
        super().__init__()
        self.num_vars = num_vars
        self.in_features = in_features

        # --- Coarse encoder: patch_size = 64 with LoRA r=lora_r ---
        enc_coarse = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=64, context_length=36, patch_size=64,
            num_samples=100, target_dim=num_vars,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        self.encoder_coarse = get_peft_model(enc_coarse, LoraConfig(
            r=lora_r, lora_alpha=lora_r * 2, target_modules="all-linear",
        ))

        # --- Fine encoder: patch_size = 8 fully fine-tuned (no LoRA) ---
        self.encoder_fine = MoiraiEncoder(
            module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{size}"),
            prediction_length=8, context_length=36, patch_size=8,
            num_samples=100, target_dim=num_vars,
            feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
        )
        # All parameters of the fine encoder are trainable by default

        # --- Cross-attention layer (coarse query → fine key/value) ---
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=in_features,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True,
        )

        # --- Classifier head ---
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(num_vars * in_features, num_classes)

    def forward(self, target, obs, pad):
        B = target.shape[0]
        F = self.in_features

        # 1. Coarse features → keep first patch per variable as query
        Z_coarse = self.encoder_coarse(target, obs, pad)        # (B, num_vars*P_coarse, F)
        P_coarse = Z_coarse.shape[1] // self.num_vars
        Z_coarse = Z_coarse.view(B, self.num_vars, P_coarse, F)
        query = Z_coarse[:, :, 0, :]                            # (B, num_vars, F)

        # 2. Fine features → all patches as key/value
        Z_fine = self.encoder_fine(target, obs, pad)            # (B, num_vars*P_fine, F)
        P_fine = Z_fine.shape[1] // self.num_vars
        Z_fine = Z_fine.view(B, self.num_vars, P_fine, F)

        # 3. Cascade cross-attention
        q  = query.view(B * self.num_vars, 1, F)
        kv = Z_fine.view(B * self.num_vars, P_fine, F)

        enriched, _ = self.cross_attn(query=q, key=kv, value=kv)
        enriched = enriched.squeeze(1).view(B, self.num_vars, F)
        enriched = enriched + query                             # residual

        # 4. Classify
        out = enriched.view(B, -1)
        return self.classifier(self.dropout(out))

## Repeated Experiment (5 seeds × 3 models)

In [42]:
MODEL_NAMES = [
    "FullFT p8 + MeanPooling",
    "FullFT p8 + MHA (4 heads)",
    "FullFT p8 + LoRA p64 + MeanPool Concat",
    "LoRA p64 + FullFT p8 + Cascade",
]

METRICS_TO_TRACK = ["Accuracy", "Macro F1", "Macro Recall"]

# results[model_name][metric] = list of values across seeds
results = {name: {m: [] for m in METRICS_TO_TRACK} for name in MODEL_NAMES}

print(f"Starting {N_REPEATS} repetitions for {len(MODEL_NAMES)} models...\n")

for run_idx, seed in enumerate(SEEDS):
    print(f"{'='*60}")
    print(f"RUN {run_idx + 1}/{N_REPEATS}  —  seed={seed}")
    print(f"{'='*60}")
    set_seed(seed)

    # --------------------------------------------------------
    # Model 1: FullFT p8 + Mean Pooling
    # --------------------------------------------------------
    print(f"\n[{MODEL_NAMES[0]}]")
    _, metrics1 = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": 8, "num_vars": NUM_VARS, "size": SIZE,
            "remove_last_patch": False,
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader,
        device=DEVICE, lr_grid=lr_grid, wd_grid=wd_grid, verbose=True,
    )
    for m in METRICS_TO_TRACK:
        results[MODEL_NAMES[0]][m].append(metrics1[m])
    print(f"  → Accuracy: {metrics1['Accuracy']:.2f}")

    # --------------------------------------------------------
    # Model 2: FullFT p8 + MHA (4 heads)
    # --------------------------------------------------------
    print(f"\n[{MODEL_NAMES[1]}]")
    _, metrics2 = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": SingleScaleMultiHeadClassifier,
            "head_kwargs": {
                "num_vars": NUM_VARS,
                "num_classes": num_classes,
                "num_heads": NUM_HEADS_MHA,
                "patch_mode": "independent_context",
            },
            "patch_size": 8, "num_vars": NUM_VARS, "size": SIZE,
            "remove_last_patch": False,
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader,
        device=DEVICE, lr_grid=lr_grid, wd_grid=wd_grid, verbose=True,
    )
    for m in METRICS_TO_TRACK:
        results[MODEL_NAMES[1]][m].append(metrics2[m])
    print(f"  → Accuracy: {metrics2['Accuracy']:.2f}")

    # --------------------------------------------------------
    # Model 3: FullFT p8 + LoRA p64 — MeanPool + Concat
    # --------------------------------------------------------
    print(f"\n[{MODEL_NAMES[2]}]")
    _, metrics3 = universal_grid_search(
        model_class=DualHybridMeanPoolWrapper,
        model_kwargs={
            "num_classes": num_classes,
            "num_vars": NUM_VARS,
            "size": SIZE,
            "lora_r": LORA_R,
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader,
        device=DEVICE, lr_grid=lr_grid, wd_grid=wd_grid, verbose=True,
    )
    for m in METRICS_TO_TRACK:
        results[MODEL_NAMES[2]][m].append(metrics3[m])
    print(f"  → Accuracy: {metrics3['Accuracy']:.2f}")

    # --------------------------------------------------------
    # Model 4: LoRA p64 (coarse) + FullFT p8 (fine) + Cascade
    # --------------------------------------------------------
    print(f"\n[{MODEL_NAMES[3]}]")
    _, metrics4 = universal_grid_search(
        model_class=DualHybridCoarseToFineWrapper,
        model_kwargs={
            "num_classes": num_classes,
            "num_vars": NUM_VARS,
            "size": SIZE,
            "lora_r": LORA_R,
            "num_heads": NUM_HEADS_MHA,
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader,
        device=DEVICE, lr_grid=lr_grid, wd_grid=wd_grid, verbose=True,
    )
    for m in METRICS_TO_TRACK:
        results[MODEL_NAMES[3]][m].append(metrics4[m])
    print(f"  → Accuracy: {metrics4['Accuracy']:.2f}")

print("\nAll runs completed.")

Starting 10 repetitions for 4 models...

RUN 1/10  —  seed=42

[FullFT p8 + MeanPooling]
LR=5e-05| WD=0.05
1.936730434254902
1.6750284917955476
1.4985762359650154
1.3753993113835652
1.2874951973194029
1.2412117613040334
1.1868459567791079
1.1865920768520697
1.1640174728098924
1.1308708278144277
1.1115070261606357
1.1115698077814367
1.1089194218317668
1.1278257418454178
1.1478497497434539
1.170529021480219
1.1958742422786186
1.211799458759587
 Early stopping : 17
Val Loss: 1.1089
LR=5e-05| WD=0.01
2.0028622886998866
1.7107331733393476
1.5109229145980463
1.3763693794002378
1.288261051100444
1.2399833541575487
1.216912991632291
1.1930478491434238
1.1536901772506838
1.1646872671639048
1.1479945105265796
1.1624512720883378
1.1892920112222192
1.1855764970546816
1.2463673013981764
1.2761251945805743
 Early stopping : 15
Val Loss: 1.1480
  → Accuracy: 0.65

[FullFT p8 + MHA (4 heads)]
LR=5e-05| WD=0.05
2.081220132548635
1.8014864766500829
1.6077569975116388
1.4281978064436254
1.335244545122472

## Results Summary

In [43]:
# --- Per-run accuracy table ---
df_runs = pd.DataFrame(
    {name: results[name]["Accuracy"] for name in MODEL_NAMES},
    index=[f"Seed {s}" for s in SEEDS],
)
df_runs.columns.name = "Model"
df_runs.index.name = "Run"

print("Per-run Test Accuracy:")
display(df_runs.round(2))

# --- Mean ± Std + Min/Max summary across all metrics ---
summary_rows = []
for name in MODEL_NAMES:
    row = {"Model": name}
    for m in METRICS_TO_TRACK:
        vals = np.array(results[name][m])
        row[f"{m} mean"] = vals.mean()
        row[f"{m} std"]  = vals.std()
        row[f"{m} min"]  = vals.min()
        row[f"{m} max"]  = vals.max()
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Model")

print("\nMean ± Std across 10 seeds:")
display(df_summary.round(2))

# Compact accuracy view
print("\nAccuracy — mean ± std [min, max] (compact):")
for name in MODEL_NAMES:
    mu  = df_summary.loc[name, "Accuracy mean"]
    std = df_summary.loc[name, "Accuracy std"]
    mn  = df_summary.loc[name, "Accuracy min"]
    mx  = df_summary.loc[name, "Accuracy max"]
    print(f"  {name:<45}: {mu:.3f} ± {std:.3f}  [min={mn:.3f}, max={mx:.3f}]")

Per-run Test Accuracy:


Model,FullFT p8 + MeanPooling,FullFT p8 + MHA (4 heads),FullFT p8 + LoRA p64 + MeanPool Concat,LoRA p64 + FullFT p8 + Cascade
Run,,,,
Seed 42,0.65,0.65,0.64,0.64
Seed 1337,0.65,0.64,0.63,0.65
Seed 0,0.66,0.64,0.66,0.64
Seed 7,0.63,0.63,0.65,0.63
Seed 99,0.65,0.65,0.65,0.63
Seed 123,0.64,0.65,0.65,0.65
Seed 256,0.64,0.63,0.65,0.65
Seed 512,0.63,0.64,0.65,0.62
Seed 2024,0.64,0.63,0.64,0.63



Mean ± Std across 10 seeds:


,Accuracy mean,Accuracy std,Accuracy min,Accuracy max,Macro F1 mean,Macro F1 std,Macro F1 min,Macro F1 max,Macro Recall mean,Macro Recall std,Macro Recall min,Macro Recall max
Model,,,,,,,,,,,,
FullFT p8 + MeanPooling,0.64,0.01,0.63,0.66,0.38,0.01,0.36,0.39,0.38,0.01,0.36,0.40
FullFT p8 + MHA (4 heads),0.64,0.01,0.63,0.65,0.37,0.01,0.34,0.38,0.37,0.01,0.34,0.39
FullFT p8 + LoRA p64 + MeanPool Concat,0.65,0.01,0.63,0.66,0.38,0.01,0.36,0.39,0.38,0.01,0.36,0.39
LoRA p64 + FullFT p8 + Cascade,0.64,0.01,0.62,0.65,0.37,0.02,0.33,0.39,0.37,0.01,0.34,0.39



Accuracy — mean ± std [min, max] (compact):
  FullFT p8 + MeanPooling                      : 0.643 ± 0.009  [min=0.627, max=0.657]
  FullFT p8 + MHA (4 heads)                    : 0.640 ± 0.007  [min=0.630, max=0.652]
  FullFT p8 + LoRA p64 + MeanPool Concat       : 0.647 ± 0.006  [min=0.635, max=0.659]
  LoRA p64 + FullFT p8 + Cascade               : 0.639 ± 0.011  [min=0.617, max=0.655]


In [44]:
# --- Tableau récapitulatif : mean / std / min / max ---
rows = []
for name in MODEL_NAMES:
    row = {"Model": name}
    for m in METRICS_TO_TRACK:
        vals = np.array(results[name][m])
        row[f"{m}\nmean"] = f"{vals.mean():.3f}"
        row[f"{m}\nstd"]  = f"{vals.std():.3f}"
        row[f"{m}\nmin"]  = f"{vals.min():.3f}"
        row[f"{m}\nmax"]  = f"{vals.max():.3f}"
    rows.append(row)

df_table = pd.DataFrame(rows).set_index("Model")
display(df_table)

,Accuracy\nmean,Accuracy\nstd,Accuracy\nmin,Accuracy\nmax,Macro F1\nmean,Macro F1\nstd,Macro F1\nmin,Macro F1\nmax,Macro Recall\nmean,Macro Recall\nstd,Macro Recall\nmin,Macro Recall\nmax
Model,,,,,,,,,,,,
FullFT p8 + MeanPooling,0.643,0.009,0.627,0.657,0.376,0.011,0.356,0.393,0.378,0.012,0.362,0.400
FullFT p8 + MHA (4 heads),0.640,0.007,0.630,0.652,0.370,0.011,0.344,0.385,0.373,0.013,0.343,0.392
FullFT p8 + LoRA p64 + MeanPool Concat,0.647,0.006,0.635,0.659,0.380,0.009,0.364,0.389,0.380,0.009,0.364,0.388
LoRA p64 + FullFT p8 + Cascade,0.639,0.011,0.617,0.655,0.370,0.016,0.332,0.389,0.372,0.015,0.341,0.394


## Classification Report — Best Seed per Model

In [45]:
from sklearn.metrics import classification_report as skl_classification_report

SEED_REPORT = SEEDS[0]
print(f"Classification reports for seed={SEED_REPORT}\n")

MODEL_CONFIGS = [
    (MODEL_NAMES[0], FullHeadWrapper, {
        "head_class": MeanPoolingClassifier,
        "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
        "patch_size": 8, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False,
    }),
    (MODEL_NAMES[1], FullHeadWrapper, {
        "head_class": SingleScaleMultiHeadClassifier,
        "head_kwargs": {
            "num_vars": NUM_VARS, "num_classes": num_classes,
            "num_heads": NUM_HEADS_MHA, "patch_mode": "independent_context",
        },
        "patch_size": 8, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False,
    }),
    (MODEL_NAMES[2], DualHybridMeanPoolWrapper, {
        "num_classes": num_classes, "num_vars": NUM_VARS, "size": SIZE, "lora_r": LORA_R,
    }),
    (MODEL_NAMES[3], DualHybridCoarseToFineWrapper, {
        "num_classes": num_classes, "num_vars": NUM_VARS, "size": SIZE,
        "lora_r": LORA_R, "num_heads": NUM_HEADS_MHA,
    }),
]

os.makedirs("results_csv", exist_ok=True)

for name, model_class, model_kwargs in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"{name}  |  seed={SEED_REPORT}")
    print(f"{'='*60}")
    set_seed(SEED_REPORT)

    best_model, _ = universal_grid_search(
        model_class=model_class, model_kwargs=model_kwargs,
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader,
        device=DEVICE, lr_grid=lr_grid, wd_grid=wd_grid, verbose=False,
    )

    best_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for b_t, b_o, b_p, b_y in te_loader:
            b_t, b_o, b_p = b_t.to(DEVICE), b_o.to(DEVICE), b_p.to(DEVICE)
            preds = torch.argmax(best_model(b_t, b_o, b_p), dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(b_y.cpu().numpy())

    print(skl_classification_report(all_targets, all_preds, digits=2))

    df_report = pd.DataFrame(
        skl_classification_report(all_targets, all_preds, output_dict=True)
    ).T
    safe_name = name.replace(" ", "_").replace("+", "").replace("(", "").replace(")", "")
    df_report.to_csv(f"results_csv/nb07_report_{safe_name}.csv")
    print(f"Saved results_csv/nb07_report_{safe_name}.csv")

Classification reports for seed=42


FullFT p8 + MeanPooling  |  seed=42
LR=5e-05| WD=0.05
 Early stopping : 16
Val Loss: 1.1031
LR=5e-05| WD=0.01
 Early stopping : 16
Val Loss: 1.1088


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

              precision    recall  f1-score   support

           0       0.67      0.36      0.47       124
           1       0.93      0.93      0.93       270
           2       0.53      0.29      0.37       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       0.50      0.03      0.05        35
           6       0.23      0.15      0.18       153
           7       0.00      0.00      0.00        24
           8       0.81      0.91      0.86       313
           9       0.00      0.00      0.00        68
          10       0.87      0.98      0.92       121
          11       0.57      0.91      0.70       777
          12       0.79      0.75      0.77        77
          13       0.50      0.13      0.21        52

    accuracy                           0.65      2466
   macro avg       0.46      0.39      0.39      2466
weighted avg       0.60      0.65      0.60      2466

Saved results_csv/nb07_re

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

              precision    recall  f1-score   support

           0       0.62      0.53      0.57       124
           1       0.85      0.96      0.90       270
           2       0.45      0.35      0.39       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       0.00      0.00      0.00        35
           6       0.30      0.14      0.19       153
           7       0.00      0.00      0.00        24
           8       0.74      0.89      0.81       313
           9       0.00      0.00      0.00        68
          10       0.85      0.93      0.89       121
          11       0.60      0.86      0.71       777
          12       0.74      0.45      0.56        77
          13       0.33      0.13      0.19        52

    accuracy                           0.64      2466
   macro avg       0.39      0.38      0.37      2466
weighted avg       0.57      0.64      0.59      2466

Saved results_csv/nb07_re

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

              precision    recall  f1-score   support

           0       0.73      0.27      0.39       124
           1       0.94      0.89      0.91       270
           2       0.44      0.44      0.44       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       0.00      0.00      0.00        35
           6       0.20      0.01      0.01       153
           7       0.00      0.00      0.00        24
           8       0.78      0.87      0.82       313
           9       0.00      0.00      0.00        68
          10       0.88      0.97      0.92       121
          11       0.57      0.88      0.69       777
          12       0.65      0.84      0.73        77
          13       0.67      0.12      0.20        52

    accuracy                           0.64      2466
   macro avg       0.42      0.38      0.37      2466
weighted avg       0.58      0.64      0.58      2466

Saved results_csv/nb07_re

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

In [46]:
# --- Export to CSV ---
import os
os.makedirs("results_csv", exist_ok=True)

rows_csv = []
for name in MODEL_NAMES:
    for seed_idx, seed in enumerate(SEEDS):
        rows_csv.append({
            "model": name,
            "seed": seed,
            "accuracy": results[name]["Accuracy"][seed_idx],
            "macro_f1": results[name]["Macro F1"][seed_idx],
            "macro_recall": results[name]["Macro Recall"][seed_idx],
        })

df_csv = pd.DataFrame(rows_csv)
df_csv.to_csv("results_csv/nb07_results.csv", index=False
)
print("Saved results_csv/nb07_results.csv")
display(df_csv)

Saved results_csv/nb07_results.csv


,model,seed,accuracy,macro_f1,macro_recall
0,FullFT p8 + MeanPooling,42,0.648013,0.376552,0.371494
1,FullFT p8 + MeanPooling,1337,0.653690,0.392382,0.396046
2,FullFT p8 + MeanPooling,0,0.656934,0.380641,0.385294
3,FullFT p8 + MeanPooling,7,0.633414,0.367442,0.370983
4,FullFT p8 + MeanPooling,99,0.648824,0.373488,0.372580
5,FullFT p8 + MeanPooling,123,0.642336,0.371303,0.376826
6,FullFT p8 + MeanPooling,256,0.644363,0.392728,0.399912
7,FullFT p8 + MeanPooling,512,0.627332,0.368105,0.369092
8,FullFT p8 + MeanPooling,2024,0.640714,0.377432,0.375679
9,FullFT p8 + MeanPooling,314,0.639092,0.356490,0.361531
